# Chess Move Prediction Model Training

This notebook loads chess game data, preprocesses it, trains a Convolutional Neural Network (CNN) to predict the next move based on the board state (FEN), and saves the trained model.

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import chess
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical
import os
print(f"TensorFlow Version: {tf.__version__}")

TensorFlow Version: 2.19.0


## 2. Helper Functions

In [2]:
def fen_to_onehot(fen: str) -> np.ndarray | None:
    """
    Converts a FEN string to a numpy array of shape (8,8,12) representing the board state.
    Returns None if the FEN is invalid.
    """
    try:
        board = chess.Board(fen)
    except ValueError:
        # Handle cases where the FEN string itself is invalid according to python-chess
        return None
        
    onehot = np.zeros((8,8,12), dtype=np.uint8)
    mapping = {
        chess.PAWN: 0,
        chess.KNIGHT: 1,
        chess.BISHOP: 2,
        chess.ROOK: 3,
        chess.QUEEN: 4,
        chess.KING: 5
    }

    for square in chess.SQUARES:
        piece = board.piece_at(square)
        if piece:
            rank = chess.square_rank(square) # 0-7
            file = chess.square_file(square) # 0-7
            # Map piece type and color to the correct channel (0-5 for white, 6-11 for black)
            channel = mapping[piece.piece_type] + (0 if piece.color == chess.WHITE else 6)
            onehot[rank, file, channel] = 1 # Use rank/file directly for indexing

    return onehot

def uci_to_index(uci: str) -> int | None:
    """
    Converts a UCI move string (e.g., 'e2e4') to an index in the range [0, 4095].
    Returns None if the UCI string is invalid.
    Mapping: index = start_square * 64 + end_square
    """
    if not isinstance(uci, str) or len(uci) < 4:
        return None
    try:
        # Basic validation for UCI format (e.g., 'a1h8')
        start_file = ord(uci[0]) - ord('a')
        start_rank = int(uci[1]) - 1
        end_file = ord(uci[2]) - ord('a')
        end_rank = int(uci[3]) - 1
        
        if not (0 <= start_file <= 7 and 0 <= start_rank <= 7 and \
                0 <= end_file <= 7 and 0 <= end_rank <= 7):
            return None # Invalid square coordinates
            
        start_square = 8 * start_rank + start_file # Square index 0-63
        end_square = 8 * end_rank + end_file     # Square index 0-63
        
        # The index represents the move from start_square to end_square
        # There are 64 possible start squares and 64 possible end squares
        # We map this to a single index: start_square * 64 + end_square
        return start_square * 64 + end_square
    except (ValueError, IndexError):
        return None # Handle parsing errors

## 3. Load and Preprocess Data

In [ ]:
data_path = 'training_data.csv'
if not os.path.exists(data_path):
    print(f"Error: Training data file not found at {data_path}")
else:
    print(f"Loading data from {data_path}...")
    df = pd.read_csv(data_path)
    print(f"Loaded {len(df)} records.")
    
    
    # df_sampled = df.sample(frac=0.1, random_state=42)
    df_sampled = df # Using all data for now
    print(f"Using {len(df_sampled)} records for processing.")

    # --- Data Conversion with Error Handling ---
    print("Converting FEN and UCI... This may take a while.")
    X_list = []
    y_list = []
    invalid_fen_count = 0
    invalid_uci_count = 0

    # Ensure columns exist
    if 'Board' not in df_sampled.columns or 'Move' not in df_sampled.columns:
        print("Error: Required columns 'Board' or 'Move' not found in CSV.")
    else:
        for index, row in df_sampled.iterrows():
            fen_str = row['Board']
            uci_str = row['Move']
            
            onehot_board = fen_to_onehot(fen_str)
            move_index = uci_to_index(uci_str)
            
            if onehot_board is None:
                invalid_fen_count += 1
                continue # Skip this row
            if move_index is None:
                invalid_uci_count += 1
                continue # Skip this row
                
            X_list.append(onehot_board)
            y_list.append(move_index)
            
            # Print progress occasionally
            if (index + 1) % 10000 == 0:
                print(f"Processed {index + 1}/{len(df_sampled)} records...")

    print("--- Processing Summary ---")
    print(f"Successfully processed: {len(X_list)} records")
    print(f"Skipped invalid FEN: {invalid_fen_count} records")
    print(f"Skipped invalid UCI: {invalid_uci_count} records")
    print("-------------------------")

    if not X_list:
        print("Error: No valid data processed. Cannot train model.")
    else:
        # Convert lists to numpy arrays
        X_train = np.array(X_list)
        y_train_indices = np.array(y_list)
        
        # One-hot encode the move indices
        # The number of classes is 4096 (64 start squares * 64 end squares)
        num_classes = 4096 
        y_train = to_categorical(y_train_indices, num_classes=num_classes)
        
        print(f"\nFinal Training Data Shapes:")
        print(f"X_train shape: {X_train.shape}")
        print(f"y_train shape: {y_train.shape}")

Loading data from training_data.csv...
Loaded 1212827 records.
Using 1212827 records for processing.
Converting FEN and UCI... This may take a while.
Processed 10000/1212827 records...
Processed 20000/1212827 records...
Processed 30000/1212827 records...
Processed 40000/1212827 records...
Processed 50000/1212827 records...
Processed 60000/1212827 records...
Processed 70000/1212827 records...
Processed 80000/1212827 records...
Processed 90000/1212827 records...
Processed 100000/1212827 records...
Processed 110000/1212827 records...
Processed 120000/1212827 records...
Processed 130000/1212827 records...
Processed 140000/1212827 records...
Processed 150000/1212827 records...
Processed 160000/1212827 records...
Processed 170000/1212827 records...
Processed 180000/1212827 records...
Processed 190000/1212827 records...
Processed 200000/1212827 records...
Processed 210000/1212827 records...
Processed 220000/1212827 records...
Processed 230000/1212827 records...
Processed 240000/1212827 record

## 4. Define Model Architecture

In [ ]:
def build_model(input_shape=(8, 8, 12), num_classes=4096):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        # layers.BatchNormalization(), Can help stabilize training
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        # layers.BatchNormalization(),
        # layers.MaxPooling2D((2, 2)), 
        layers.Flatten(),
        layers.Dense(512, activation='relu'),
        # layers.Dropout(0.5), # Optional: Regularization
        layers.Dense(num_classes, activation='softmax') # Output layer for move probabilities
    ])
    return model

# Build the model only if data is available
if 'X_train' in locals() and X_train.size > 0:
    model = build_model()
    print("Model built successfully.")
    model.summary()
else:
    print("Skipping model build because training data is not available.")

Model built successfully.


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 8, 8, 32)       │         3,488 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 8, 8, 64)       │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │     2,097,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4096)           │     2,101,248 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,220,896 (16.10 MB)

 Trainable params: 4,220,896 (16.10 MB)

 Non-trainable params: 0 (0.00 B)

## 5. Compile Model

In [5]:
# Compile the model only if it was built
if 'model' in locals():
    model.compile(optimizer='adam', 
                  loss='categorical_crossentropy', 
                  metrics=['accuracy'])
    print("Model compiled successfully.")
else:
    print("Skipping model compilation because the model was not built.")

Model compiled successfully.


## 6. Train Model

In [6]:
# Train the model only if data and model are ready
if 'model' in locals() and 'X_train' in locals() and X_train.size > 0:
    print("Starting model training...")
    history = model.fit(X_train, y_train, 
                        batch_size=64,
                        epochs=5,
                        validation_split=0.1)
    print("Model training finished.")
else:
    print("Skipping model training due to missing data or model.")

Starting model training...
Epoch 1/5
17056/17056 ━━━━━━━━━━━━━━━━━━━━ 220s 13ms/step - accuracy: 0.1198 - loss: 5.0645 - val_accuracy: 0.1869 - val_loss: 3.8134
Epoch 2/5
17056/17056 ━━━━━━━━━━━━━━━━━━━━ 221s 13ms/step - accuracy: 0.2050 - loss: 3.5830 - val_accuracy: 0.2050 - val_loss: 3.6111
Epoch 3/5
17056/17056 ━━━━━━━━━━━━━━━━━━━━ 219s 13ms/step - accuracy: 0.2325 - loss: 3.3148 - val_accuracy: 0.2140 - val_loss: 3.5497
Epoch 4/5
17056/17056 ━━━━━━━━━━━━━━━━━━━━ 217s 13ms/step - accuracy: 0.2552 - loss: 3.1418 - val_accuracy: 0.2136 - val_loss: 3.5371
Epoch 5/5
17056/17056 ━━━━━━━━━━━━━━━━━━━━ 221s 13ms/step - accuracy: 0.2742 - loss: 3.0080 - val_accuracy: 0.2200 - val_loss: 3.5491
Model training finished.


## 7. Save Model

In [7]:
# Save the model only if training was attempted
if 'model' in locals() and 'history' in locals():
    # Ensure the target directory exists relative to the notebook location
    model_dir = '../Models' # Save in parent directory's Models folder
    os.makedirs(model_dir, exist_ok=True)
    
    # Save the model using the recommended .keras format
    model_path = os.path.join(model_dir, 'chess_model.keras')
    try:
        model.save(model_path)
        print(f'Model successfully saved to {model_path}')
    except Exception as e:
        print(f"Error saving model: {e}")
else:
    print("Skipping model saving because training was not performed.")

Model successfully saved to ../Models/chess_model.keras
